In [75]:
import tensorflow as tf
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive
from sklearn.preprocessing import StandardScaler
import joblib

Parametros globales

In [76]:
umbral = 0.8 #Hiper Parámetro
clase = 0
features = 6

Funcion para crear un nuevo modelo

In [77]:
def NewModel(umbral, clase, features):
    model = tf.keras.models.Sequential([
      tf.keras.layers.Input(shape=(6,), name='entrada'),
      tf.keras.layers.Dense(64, activation='relu', name='capa_1'),
      tf.keras.layers.Dense(32, activation='relu', name='capa_2'),
      tf.keras.layers.Dense(16, activation='relu', name='capa_3'),
      tf.keras.layers.Dense(1, activation='sigmoid', name='capa_salida')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.Recall(name='recall_phishing', thresholds=umbral, class_id=clase)]
    )
    return model

Cargar los pesos y darselos al modelo

In [78]:
drive.mount('/content/drive')
model = NewModel(umbral, clase, features)
ruta_modelo = '/content/drive/MyDrive/Colab_Notebooks/reto/Propio/Pesos_Phishing.weights.h5'
model.load_weights(ruta_modelo, skip_mismatch=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Interfaz de prueba

In [79]:



features = [
    'HTTPS',
    'AnchorURL',
    'PrefixSuffix-',
    'WebsiteTraffic',
    'SubDomains',
    'RequestURL'
]
Inputs = []
for nombre in features:
    cuadro = widgets.IntText(value=0, description=f'{nombre}:')
    Inputs.append(cuadro)
ui = widgets.GridBox(Inputs, layout=widgets.Layout(grid_template_columns="repeat(2, 300px)"))

boton_predecir = widgets.Button(description="Probar", button_style='danger')
output_resultado = widgets.Output()



In [80]:
drive.mount('/content/drive')
ruta = '/content/drive/MyDrive/Colab_Notebooks/reto/Propio/escalador_phishing.pkl'
scaler_interfaz = joblib.load(ruta)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [81]:
def realizar_prediccion(b):
    with output_resultado:
        output_resultado.clear_output()

        datos = []
        for w in Inputs:
            datos.append(w.value)


        instancia = np.array(datos).reshape(1, -1)
        instancia_escalada = scaler_interfaz.transform(instancia)
        prediccion_prob = model.predict(instancia_escalada, verbose=0)

        if prediccion_prob[0][0] > umbral:
            resultado = "LEGÍTIMO (1)"
            color = "\033[92m"
        else:
            resultado = "PHISHING (0)"
            color = "\033[91m"
        print("-------------------------------------------------------")
        print(f"Probabilidad de Legitimidad: {prediccion_prob[0][0]:.4f}")
        print(f"Resultado final: {color}{resultado}\033[0m")
        print("-------------------------------------------------------")

In [82]:

print("-------------------------- INTERFAZ DE CALIFICACIÓN DE PHISHING ----------------------------")

print("="*50)
print("   INSTRUCCIONES DE CALIFICACIÓN")
print("="*50)
print(" 1  -> Seguro / Legítimo")
print(" 0  -> Sospechoso / Neutral")
print("-1  -> Peligro / Phishing")
print("-" * 50)
print("Parámetros clave:")
print(" - HTTPS: Seguridad de la conexión.")
print(" - AnchorURL: Destino de los enlaces internos.")
print(" - PrefixSuffix: Guiones en el nombre del dominio.")
print(" - WebsiteTraffic: Popularidad del sitio.")
print(" - SubDomains: Cantidad de puntos en la URL.")
print(" - RequestURL: Origen de los objetos externos.")
print("="*50)
print(f"{'                  '} | {'-1':<15} | {'0':<15} | {'1':<15}")
print("-"*80)
print(f"{'1. HTTPS':<18} | {'Sin SSL/HTTP':<15} | {'Cert. No Ofic.':<15} | {'Cert. Válido':<15}")
print(f"{'2. AnchorURL':<18} | {'Links ext >67%':<15} | {'Links ext 31-67%':<15} | {'Links ext <31%':<15}")
print(f"{'3. PrefixSuffix':<18} | {'Tiene guiones':<15} | {'---':<15} | {'Sin guiones':<15}")
print(f"{'4. Traffic':<18} | {'Sin tráfico':<15} | {'Tráfico Medio':<15} | {'Muy visitado':<15}")
print(f"{'5. SubDomains':<18} | {'Puntos >2':<15} | {'Puntos = 2':<15} | {'Puntos < 2':<15}")
print(f"{'6. RequestURL':<18} | {'Objetos ext >61%':<15} | {'Obj. ext 22-61%':<15} | {'Obj. ext <22%':<15}")

print("-" * 65)
print("Selecciona el valor numérico corresponiente.")
print("="*65)
print("\n")

# Aquí va tu display(ui, boton_predecir, ...)


boton_predecir.on_click(realizar_prediccion)

display(ui, boton_predecir, output_resultado)



-------------------------- INTERFAZ DE CALIFICACIÓN DE PHISHING ----------------------------
   INSTRUCCIONES DE CALIFICACIÓN
 1  -> Seguro / Legítimo
 0  -> Sospechoso / Neutral
-1  -> Peligro / Phishing
--------------------------------------------------
Parámetros clave:
 - HTTPS: Seguridad de la conexión.
 - AnchorURL: Destino de los enlaces internos.
 - PrefixSuffix: Guiones en el nombre del dominio.
 - WebsiteTraffic: Popularidad del sitio.
 - SubDomains: Cantidad de puntos en la URL.
 - RequestURL: Origen de los objetos externos.
                   | -1              | 0               | 1              
--------------------------------------------------------------------------------
1. HTTPS           | Sin SSL/HTTP    | Cert. No Ofic.  | Cert. Válido   
2. AnchorURL       | Links ext >67%  | Links ext 31-67% | Links ext <31% 
3. PrefixSuffix    | Tiene guiones   | ---             | Sin guiones    
4. Traffic         | Sin tráfico     | Tráfico Medio   | Muy visitado   
5. SubDomai

GridBox(children=(IntText(value=0, description='HTTPS:'), IntText(value=0, description='AnchorURL:'), IntText(…

Button(button_style='danger', description='Probar', style=ButtonStyle())

Output()